# TP 3 : Pipeline RAG Simple

## Objectif

Construire un système RAG (Retrieval-Augmented Generation) complet pour répondre à des questions sur des documents, en intégrant le chunking, l'indexation vectorielle et la génération de réponses.

---
## Partie 1 : Installation et Configuration 

### Tâche 1.1 : Installer les dépendances

In [ ]:
# TODO : Installer les bibliothèques nécessaires
# - langchain
# - langchain-community
# - chromadb
# - sentence-transformers
# - pypdf (pour lire les PDFs)
# - openai (optionnel, pour la génération)

# !pip install langchain langchain-community chromadb sentence-transformers pypdf openai

### Tâche 1.2 : Importer les bibliothèques

In [ ]:
# TODO : Importer toutes les bibliothèques nécessaires
# - langchain text_splitter (RecursiveCharacterTextSplitter)
# - langchain document_loaders (TextLoader, PyPDFLoader, DirectoryLoader)
# - chromadb
# - sentence_transformers

# Votre code ici

---
## Partie 2 : Préparation des Documents 

### Tâche 2.1 : Créer des documents d'exemple

Nous allons créer 3-5 fichiers texte sur différents sujets techniques.

In [1]:
# Créer un répertoire pour les documents
import os

docs_dir = "./tp3_documents"
os.makedirs(docs_dir, exist_ok=True)

# Documents d'exemple
documents = {
    "python_basics.txt": """
# Python : Les Fondamentaux

Python est un langage de programmation interprété, orienté objet et de haut niveau créé par Guido van Rossum en 1991.
Il se distingue par sa syntaxe claire et lisible qui favorise la productivité des développeurs.

## Types de Données

Python propose plusieurs types de données natifs :
- **Numériques** : int (entiers), float (nombres à virgule), complex (nombres complexes)
- **Séquences** : str (chaînes), list (listes), tuple (tuples immuables)
- **Mappings** : dict (dictionnaires)
- **Ensembles** : set (ensembles), frozenset (ensembles immuables)
- **Booléens** : bool (True/False)

## Structures de Contrôle

Les structures de contrôle en Python incluent :
- **Conditionnelles** : if, elif, else
- **Boucles** : for (itération), while (condition)
- **Gestion d'erreurs** : try, except, finally

## Fonctions

Les fonctions en Python sont définies avec le mot-clé `def`. Elles peuvent accepter des arguments positionnels,
des arguments nommés, des arguments par défaut, et même un nombre variable d'arguments (*args, **kwargs).

Exemple :
```python
def calculer_moyenne(*nombres):
    return sum(nombres) / len(nombres) if nombres else 0
```

## Modules et Packages

Python dispose d'un vaste écosystème de bibliothèques. La bibliothèque standard inclut des modules pour :
- Manipulation de fichiers (os, pathlib)
- Calculs mathématiques (math, statistics)
- Dates et heures (datetime)
- Expressions régulières (re)
- Requêtes HTTP (urllib, http)
""",
    
    "machine_learning.txt": """
# Introduction au Machine Learning

Le Machine Learning (apprentissage automatique) est une branche de l'intelligence artificielle qui permet
aux ordinateurs d'apprendre à partir de données sans être explicitement programmés pour chaque tâche.

## Types d'Apprentissage

### Apprentissage Supervisé

Dans l'apprentissage supervisé, le modèle est entraîné sur des données étiquetées. Les algorithmes courants incluent :
- **Régression Linéaire** : prédiction de valeurs continues
- **Régression Logistique** : classification binaire
- **Arbres de Décision** : classification et régression
- **Random Forest** : ensemble d'arbres de décision
- **SVM (Support Vector Machines)** : classification avec marge maximale
- **Réseaux de Neurones** : modèles inspirés du cerveau humain

### Apprentissage Non Supervisé

L'apprentissage non supervisé travaille avec des données non étiquetées :
- **K-Means** : clustering par centres
- **DBSCAN** : clustering basé sur la densité
- **PCA (Analyse en Composantes Principales)** : réduction de dimensionnalité
- **Autoencoders** : apprentissage de représentations

### Apprentissage par Renforcement

Un agent apprend à prendre des décisions en interagissant avec un environnement et en recevant des récompenses.
Algorithmes populaires : Q-Learning, Deep Q-Networks (DQN), Policy Gradients.

## Bibliothèques Python

Les principales bibliothèques Python pour le ML sont :
- **scikit-learn** : algorithmes classiques de ML
- **TensorFlow** : deep learning développé par Google
- **PyTorch** : deep learning développé par Facebook
- **Keras** : API haut niveau pour les réseaux de neurones
- **XGBoost** : gradient boosting optimisé

## Workflow Typique

1. **Collecte de données** : rassembler les données pertinentes
2. **Exploration (EDA)** : analyser et visualiser les données
3. **Prétraitement** : nettoyage, normalisation, feature engineering
4. **Séparation** : train/validation/test sets
5. **Entraînement** : ajuster le modèle sur les données d'entraînement
6. **Évaluation** : mesurer les performances sur les données de test
7. **Optimisation** : hyperparamètres, architecture
8. **Déploiement** : mise en production du modèle
""",
    
    "rag_systems.txt": """
# RAG : Retrieval-Augmented Generation

RAG (Retrieval-Augmented Generation) est une architecture qui combine la recherche d'information (retrieval)
avec la génération de texte par des modèles de langage (generation).

## Principe de Fonctionnement

Un système RAG fonctionne en plusieurs étapes :

1. **Indexation** (offline) :
   - Découpage des documents en chunks
   - Génération d'embeddings vectoriels
   - Stockage dans une base de données vectorielle

2. **Recherche** (online) :
   - Conversion de la question en embedding
   - Recherche des documents les plus pertinents
   - Récupération du contexte

3. **Génération** (online) :
   - Construction d'un prompt avec le contexte
   - Génération de la réponse par le LLM
   - Post-traitement et validation

## Composants Clés

### Chunking (Découpage)

Le découpage des documents est crucial pour la performance :
- **Taille des chunks** : généralement 500-1000 tokens
- **Overlap** : chevauchement de 10-20% pour la continuité
- **Méthodes** : par caractères, par phrases, par paragraphes, sémantique

### Embeddings

Les modèles d'embedding transforment le texte en vecteurs :
- **OpenAI Embeddings** : text-embedding-3-small/large
- **Open Source** : all-MiniLM-L6-v2, BGE, E5
- **Multilingues** : multilingual-e5-large

### Bases Vectorielles

Stockage et recherche efficace des embeddings :
- **ChromaDB** : simple et embarquée
- **Pinecone** : cloud, scalable
- **Weaviate** : open source avec fonctionnalités avancées
- **FAISS** : bibliothèque Facebook, très rapide
- **Qdrant** : haute performance avec filtrage

### LLMs (Modèles de Langage)

Génération de réponses contextualisées :
- **Propriétaires** : GPT-4, Claude, Gemini
- **Open Source** : Llama 2/3, Mistral, Mixtral
- **Spécialisés** : modèles fine-tunés pour domaines spécifiques

## Techniques Avancées

### Hybrid Search

Combinaison de recherche vectorielle et keyword-based (BM25) pour améliorer la pertinence.

### Reranking

Utilisation d'un modèle de reranking (ex: cross-encoders) pour affiner les résultats de recherche.

### Query Transformation

Amélioration des requêtes :
- **HyDE** : génération de documents hypothétiques
- **Multi-Query** : génération de plusieurs variations de la question
- **Step-back** : questions plus générales pour contexte additionnel

### Metadata Filtering

Utilisation de métadonnées pour filtrer les résultats (date, catégorie, source, etc.).

## Métriques d'Évaluation

### Retrieval
- **Recall@K** : proportion de documents pertinents récupérés
- **Precision@K** : proportion de documents récupérés qui sont pertinents
- **MRR (Mean Reciprocal Rank)** : rang moyen du premier document pertinent

### Generation
- **Faithfulness** : fidélité au contexte récupéré
- **Answer Relevancy** : pertinence de la réponse à la question
- **Context Relevancy** : pertinence du contexte récupéré

## Cas d'Usage

- **Documentation technique** : réponses sur documentation produit
- **Support client** : chatbots avec base de connaissances
- **Recherche juridique** : recherche dans corpus de lois
- **Médical** : aide au diagnostic basée sur littérature médicale
- **Éducation** : assistants pédagogiques personnalisés
""",
    
    "docker_kubernetes.txt": """
# Docker et Kubernetes : Guide Pratique

## Docker

Docker est une plateforme de conteneurisation qui permet d'empaqueter des applications avec toutes leurs dépendances.

### Concepts Fondamentaux

**Image** : Template immuable contenant l'application et ses dépendances.
- Créée à partir d'un Dockerfile
- Stockée dans un registry (Docker Hub, etc.)
- Versionnable avec des tags

**Container** : Instance en cours d'exécution d'une image.
- Isolé du système hôte
- Léger (partage le kernel)
- Éphémère par défaut

**Volume** : Stockage persistant pour les données.
- Survit à la suppression du conteneur
- Peut être partagé entre conteneurs

**Network** : Réseau virtuel pour la communication entre conteneurs.

### Commandes Docker Essentielles

```bash
# Images
docker build -t mon-app:v1 .        # Construire une image
docker images                        # Lister les images
docker pull nginx:latest             # Télécharger une image
docker push mon-app:v1               # Pousser vers registry

# Conteneurs
docker run -d -p 8080:80 nginx       # Lancer un conteneur
docker ps                            # Lister conteneurs actifs
docker ps -a                         # Tous les conteneurs
docker stop <container-id>           # Arrêter
docker rm <container-id>             # Supprimer
docker logs <container-id>           # Voir les logs
docker exec -it <container-id> bash  # Shell interactif

# Volumes
docker volume create mon-volume
docker run -v mon-volume:/app/data nginx
```

### Dockerfile

Structure type d'un Dockerfile :
```dockerfile
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install -r requirements.txt
COPY . .
EXPOSE 8000
CMD ["python", "app.py"]
```

### Docker Compose

Orchestration multi-conteneurs avec un fichier YAML :
```yaml
version: '3.8'
services:
  web:
    build: .
    ports:
      - "8000:8000"
    volumes:
      - .:/app
    environment:
      - DEBUG=1
  db:
    image: postgres:15
    volumes:
      - postgres_data:/var/lib/postgresql/data
```

## Kubernetes

Kubernetes (K8s) est un orchestrateur de conteneurs open-source pour automatiser le déploiement,
la mise à l'échelle et la gestion d'applications conteneurisées.

### Architecture

**Control Plane** (Plan de contrôle) :
- **API Server** : point d'entrée pour toutes les opérations
- **etcd** : base de données clé-valeur pour l'état du cluster
- **Scheduler** : assigne les pods aux nodes
- **Controller Manager** : gère les contrôleurs (réplication, etc.)

**Nodes** (Nœuds de travail) :
- **kubelet** : agent sur chaque node
- **kube-proxy** : gestion du réseau
- **Container Runtime** : Docker, containerd, CRI-O

### Objets Kubernetes

**Pod** : Plus petite unité déployable, contient un ou plusieurs conteneurs.

**Deployment** : Gère le déploiement et la mise à jour des pods.
```yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: mon-app
spec:
  replicas: 3
  selector:
    matchLabels:
      app: mon-app
  template:
    metadata:
      labels:
        app: mon-app
    spec:
      containers:
      - name: app
        image: mon-app:v1
        ports:
        - containerPort: 8000
```

**Service** : Expose les pods sur le réseau.
- ClusterIP : interne au cluster
- NodePort : exposé sur un port de chaque node
- LoadBalancer : avec load balancer externe

**ConfigMap** : Configuration non sensible.

**Secret** : Données sensibles (mots de passe, tokens).

**Ingress** : Gestion du trafic HTTP/HTTPS entrant.

### Commandes kubectl

```bash
# Déploiement
kubectl apply -f deployment.yaml
kubectl get deployments
kubectl get pods
kubectl describe pod <pod-name>

# Scaling
kubectl scale deployment mon-app --replicas=5

# Mise à jour
kubectl set image deployment/mon-app app=mon-app:v2
kubectl rollout status deployment/mon-app
kubectl rollout undo deployment/mon-app

# Logs et debug
kubectl logs <pod-name>
kubectl exec -it <pod-name> -- /bin/bash

# Services
kubectl get services
kubectl port-forward service/mon-app 8080:80
```

### Bonnes Pratiques

1. **Resource Limits** : définir requests et limits pour CPU/mémoire
2. **Health Checks** : liveness et readiness probes
3. **Rolling Updates** : mises à jour progressives sans downtime
4. **Namespaces** : isolation logique des ressources
5. **RBAC** : contrôle d'accès basé sur les rôles
6. **Monitoring** : Prometheus, Grafana pour la supervision
""",
    
    "data_science.txt": """
# Data Science avec Python

La Data Science combine statistiques, programmation et expertise métier pour extraire des insights des données.

## Bibliothèques Essentielles

### NumPy

Calcul numérique avec des arrays multidimensionnels :
```python
import numpy as np
arr = np.array([1, 2, 3, 4, 5])
matrix = np.array([[1, 2], [3, 4]])
```

Opérations : broadcasting, slicing, aggregations, algèbre linéaire.

### Pandas

Manipulation et analyse de données tabulaires :
```python
import pandas as pd
df = pd.read_csv('data.csv')
df.head()
df.describe()
df.groupby('category')['value'].mean()
```

Structures : Series (1D), DataFrame (2D).
Opérations : filtrage, groupby, merge, pivot, time series.

### Matplotlib et Seaborn

Visualisation de données :
```python
import matplotlib.pyplot as plt
import seaborn as sns

# Matplotlib
plt.plot(x, y)
plt.scatter(x, y)
plt.hist(data)

# Seaborn
sns.boxplot(x='category', y='value', data=df)
sns.heatmap(correlation_matrix)
```

### Scikit-learn

Machine Learning :
```python
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
model = LinearRegression()
model.fit(X_train, y_train)
predictions = model.predict(X_test)
```

## Workflow Data Science

### 1. Acquisition des Données

Sources : CSV, bases de données, APIs, web scraping.
```python
df = pd.read_csv('data.csv')
# ou
import requests
response = requests.get('https://api.example.com/data')
```

### 2. Exploration (EDA)

Comprendre les données :
- Dimensions : shape, dtypes
- Statistiques : describe(), info()
- Valeurs manquantes : isnull().sum()
- Distributions : histogrammes, boxplots
- Corrélations : corr(), heatmap

### 3. Nettoyage

Préparer les données :
```python
# Valeurs manquantes
df.dropna()  # supprimer
df.fillna(df.mean())  # remplir

# Doublons
df.drop_duplicates()

# Types
df['date'] = pd.to_datetime(df['date'])

# Outliers
z_scores = (df['value'] - df['value'].mean()) / df['value'].std()
df = df[abs(z_scores) < 3]
```

### 4. Feature Engineering

Créer de nouvelles features :
```python
# Encodage catégoriel
df = pd.get_dummies(df, columns=['category'])

# Normalisation
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df)

# Binning
df['age_group'] = pd.cut(df['age'], bins=[0, 18, 35, 60, 100])
```

### 5. Modélisation

Entraîner et évaluer des modèles :
```python
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
```

### 6. Visualisation et Communication

Présenter les résultats :
- Graphiques interactifs : Plotly, Bokeh
- Dashboards : Streamlit, Dash
- Rapports : Jupyter Notebooks

## Outils Avancés

**Dask** : Pandas à grande échelle (parallélisation)
**PySpark** : Big Data avec Apache Spark
**MLflow** : Suivi d'expériences ML
**Great Expectations** : Validation de qualité des données
"""
}

# Écrire les fichiers
for filename, content in documents.items():
    filepath = os.path.join(docs_dir, filename)
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(content.strip())

print(f"✓ {len(documents)} documents créés dans {docs_dir}")
for filename in documents.keys():
    filepath = os.path.join(docs_dir, filename)
    size = os.path.getsize(filepath)
    print(f"  - {filename}: {size} bytes")

✓ 5 documents créés dans ./tp3_documents
  - python_basics.txt: 1551 bytes
  - machine_learning.txt: 2279 bytes
  - rag_systems.txt: 3361 bytes
  - docker_kubernetes.txt: 4576 bytes
  - data_science.txt: 3398 bytes


### Tâche 2.2 : Charger les documents

Utilisez LangChain pour charger tous les fichiers texte du répertoire.

In [ ]:
# TODO : Charger les documents avec DirectoryLoader
# Indices :
# - Utiliser DirectoryLoader avec glob="**/*.txt"
# - Utiliser TextLoader comme loader_cls
# - Appeler .load() pour charger tous les documents

# Votre code ici

---
## Partie 3 : Découpage en Chunks

### Tâche 3.1 : Découper avec une première stratégie

Utilisez `RecursiveCharacterTextSplitter` avec :
- chunk_size = 500
- chunk_overlap = 50

In [ ]:
# TODO : Créer un text splitter et découper les documents
# 1. Créer RecursiveCharacterTextSplitter
# 2. Appliquer split_documents() sur les documents chargés
# 3. Afficher le nombre de chunks obtenus

# Votre code ici

### Tâche 3.2 : Tester différentes tailles

Comparez les résultats avec chunk_size = 1000 et chunk_size = 300.

In [ ]:
# TODO : Tester différentes tailles de chunks
# Pour chaque taille (300, 500, 1000) :
# 1. Créer un splitter
# 2. Découper les documents
# 3. Comparer le nombre de chunks et leur contenu

# Votre code ici

### Tâche 3.3 : Analyser les chunks

Pour la taille choisie, analysez :
- La longueur moyenne des chunks
- La distribution des longueurs
- Des exemples de chunks

In [ ]:
# TODO : Analyser les chunks
# Calculer et afficher des statistiques

# Votre code ici

---
## Partie 4 : Indexation dans ChromaDB 

### Tâche 4.1 : Créer la collection ChromaDB

Créez une collection avec un modèle d'embedding approprié.

In [ ]:
# TODO : Créer un client ChromaDB et une collection
# 1. Créer le client
# 2. Supprimer la collection si elle existe
# 3. Créer une nouvelle collection nommée "docs_tech"

# Votre code ici

### Tâche 4.2 : Indexer les chunks

Ajoutez tous les chunks à la collection avec des métadonnées appropriées.

In [ ]:
# TODO : Indexer les chunks
# 1. Préparer les textes, IDs et métadonnées
# 2. Ajouter à la collection avec collection.add()
# 3. Vérifier le nombre de documents indexés
# Indices :
# - Les métadonnées peuvent inclure : source (nom du fichier), chunk_index

# Votre code ici

---
## Partie 5 : Création du Retriever 

### Tâche 5.1 : Implémenter une fonction de recherche

Créez une fonction qui prend une question et retourne les chunks les plus pertinents.

In [ ]:
# TODO : Créer une fonction retrieve(question, top_k=3)
# qui :
# 1. Effectue une recherche dans la collection
# 2. Retourne les résultats formatés

# Votre code ici

### Tâche 5.2 : Tester le retriever

Testez avec quelques questions pour vérifier la qualité de la recherche.

In [ ]:
# TODO : Tester avec des questions
test_questions = [
    "Comment fonctionne Docker ?",
    "Qu'est-ce que le RAG ?",
    "Quelles sont les bibliothèques Python pour le machine learning ?"
]

# Votre code ici

---
## Partie 6 : Génération de Réponses

### Tâche 6.1 : Créer un prompt template

Créez un template de prompt qui combine la question et le contexte.

In [ ]:
# TODO : Créer un template de prompt
# Le prompt doit :
# 1. Inclure le contexte récupéré
# 2. Inclure la question de l'utilisateur
# 3. Donner des instructions claires au LLM

# Votre code ici

### Tâche 6.2 : Intégrer un LLM (optionnel)

Si vous avez une clé API OpenAI, intégrez GPT pour générer des réponses.
Sinon, créez une réponse simple basée sur le contexte.

In [ ]:
# TODO : Créer une fonction answer(question) qui :
# 1. Récupère le contexte avec retrieve()
# 2. Construit le prompt
# 3. (Optionnel) Génère une réponse avec un LLM
# 4. Retourne la réponse ou le prompt construit

# Votre code ici

---
## Partie 7 : Tests et Évaluation 
### Tâche 7.1 : Créer un jeu de 10 questions

Préparez 10 questions variées couvrant différents documents.

In [ ]:
# TODO : Créer 10 questions de test
evaluation_questions = [
    # Ajoutez vos questions ici
]

# Votre code ici

### Tâche 7.2 : Évaluer le système

Pour chaque question, évaluez :
- La pertinence du contexte récupéré
- La qualité de la réponse générée

In [ ]:
# TODO : Tester et évaluer
# Pour chaque question :
# 1. Obtenir la réponse
# 2. Afficher la question, le contexte et la réponse
# 3. Évaluer manuellement la qualité

# Votre code ici